In [1]:
import os
import joblib
import numpy as np
import pandas as pd
import shap
from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.feature_selection import VarianceThreshold
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

secom = fetch_ucirepo(id=179)
df_orig = secom.data.original

# Automatically locate the target column containing -1 and 1
target_col = None
for col in df_orig.columns:
    unique_vals = set(pd.to_numeric(df_orig[col], errors="coerce").dropna().unique())
    if unique_vals == {-1.0, 1.0} or unique_vals == {-1, 1}:
        target_col = col
        break

print(f"Detected target column: {target_col}")

y_num = pd.to_numeric(df_orig[target_col], errors="coerce")
y = (y_num == 1).astype(int)

# Keep only numeric sensor features and drop the target column
X_raw = df_orig.drop(columns=[target_col]).select_dtypes(include=[np.number]).copy()
X_raw.columns = [f"Sensor_{i}" for i in range(X_raw.shape[1])]

print(f"Dataset shape: {X_raw.shape}")
print(f"Pass count: {(y == 0).sum()} | Fail count: {(y == 1).sum()}")


D:\Git\semiconductor_yield_project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Detected target column: class
Dataset shape: (1567, 590)
Pass count: 1463 | Fail count: 104


In [2]:
X_train, X_test, y_train, y_test = train_test_split(
    X_raw, y, test_size=0.2, random_state=42, stratify=y
)

# 1. Median imputation for missing sensor values
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(
    imputer.fit_transform(X_train), columns=X_train.columns, index=X_train.index
)
X_test_imp = pd.DataFrame(
    imputer.transform(X_test), columns=X_test.columns, index=X_test.index
)

# 2. Remove constant (zero-variance) sensors
selector = VarianceThreshold(threshold=0.0)
X_train_clean = pd.DataFrame(
    selector.fit_transform(X_train_imp),
    columns=X_train_imp.columns[selector.get_support()],
    index=X_train_imp.index,
)
X_test_clean = pd.DataFrame(
    selector.transform(X_test_imp),
    columns=X_train_imp.columns[selector.get_support()],
    index=X_test_imp.index,
)

print(f"Remaining active sensors: {X_train_clean.shape}")


Remaining active sensors: (1253, 474)


In [4]:
scale_weight = (y_train == 0).sum() / (y_train == 1).sum()

model = XGBClassifier(
    n_estimators=150,
    max_depth=4,
    learning_rate=0.05,
    scale_pos_weight=scale_weight,
    random_state=42,
    eval_metric="logloss",
)
model.fit(X_train_clean, y_train)

# Evaluate on held-out test set
y_pred = model.predict(X_test_clean)
y_prob = model.predict_proba(X_test_clean)[:, 1]

print(f"Test ROC-AUC: {roc_auc_score(y_test, y_prob):.3f}")
print(
    classification_report(
        y_test, y_pred, labels=[0, 1], target_names=["Pass", "Fail"], zero_division=0
    )
)

# Initialize SHAP explainer
explainer = shap.TreeExplainer(model)


Test ROC-AUC: 0.722
              precision    recall  f1-score   support

        Pass       0.94      0.99      0.96       293
        Fail       0.33      0.05      0.08        21

    accuracy                           0.93       314
   macro avg       0.63      0.52      0.52       314
weighted avg       0.90      0.93      0.90       314



In [5]:
# Select the first failing wafer in the test set
failing_wafers = X_test_clean[y_test == 1]
sample_id = failing_wafers.index[0]
sample_row = failing_wafers.loc[[sample_id]]

fail_prob = model.predict_proba(sample_row)[0, 1]
shap_vals = explainer.shap_values(sample_row)[0]

# Extract top 3 sensors pushing the wafer toward failure
top_indices = np.argsort(shap_vals)[::-1][:3]
print(f"Wafer ID {sample_id} | Predicted Fail Probability: {fail_prob:.2%}")
for idx in top_indices:
    sensor_name = X_test_clean.columns[idx]
    val = sample_row.iloc[0, idx]
    impact = shap_vals[idx]
    print(f"  - {sensor_name}: value={val:.3f} (SHAP impact: +{impact:.3f})")

# Save model bundle for Week 2 agent tools
os.makedirs("../models", exist_ok=True)
joblib.dump(
    {
        "imputer": imputer,
        "selector": selector,
        "model": model,
        "explainer": explainer,
        "feature_names": list(X_train_clean.columns),
        "X_test_clean": X_test_clean,
        "y_test": y_test,
    },
    "../models/secom_yield_bundle.joblib",
)
print("Saved models/secom_yield_bundle.joblib successfully!")


Wafer ID 154 | Predicted Fail Probability: 1.95%
  - Sensor_103: value=-0.006 (SHAP impact: +0.281)
  - Sensor_59: value=26.774 (SHAP impact: +0.182)
  - Sensor_423: value=129.467 (SHAP impact: +0.132)
Saved models/secom_yield_bundle.joblib successfully!
